# Python Analysis — NorthStar

Fleet condition, delivery duration, app events, and cross-dataset integration.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

orders     = pd.read_csv('/content/orders_cleaned.csv',     parse_dates=['order_created_at'])
deliveries = pd.read_csv('/content/deliveries_cleaned.csv', parse_dates=['dispatch_time', 'delivery_completed_at'])
vehicles   = pd.read_csv('/content/vehicles_cleaned.csv',   parse_dates=['commission_date'])
incidents  = pd.read_csv('/content/incidents_cleaned.csv',  parse_dates=['reported_at'])
hubs       = pd.read_csv('/content/hubs_cleaned.csv')
app_events = pd.read_csv('/content/app_events_cleaned.csv', parse_dates=['event_timestamp'])

print('Loaded.')

## 2. Vehicle fleet

In [ ]:
valid_battery = vehicles.dropna(subset=['battery_health_pct'])

fig, ax = plt.subplots(figsize=(10, 5))
for vtype in valid_battery['vehicle_type'].unique():
    subset = valid_battery[valid_battery['vehicle_type'] == vtype]
    ax.hist(subset['battery_health_pct'], bins=15, alpha=0.6, label=vtype)

ax.axvline(70, color='red', linestyle='--', label='70% alert')
ax.set_xlabel('Battery Health (%)')
ax.set_ylabel('Vehicles')
ax.set_title('Battery Health by Vehicle Type')
ax.legend()
plt.tight_layout()
plt.show()

below = valid_battery[valid_battery['battery_health_pct'] < 70]
print(f'Below 70%: {len(below)} of {len(valid_battery)} ({len(below)/len(valid_battery)*100:.1f}%)')

In [ ]:
maint_by_zone = pd.crosstab(vehicles['assigned_zone'], vehicles['maintenance_status'])
print(maint_by_zone)

maint_by_zone.plot(kind='bar', stacked=True, figsize=(10, 5),
                   color=['steelblue', '#FCD34D', 'tomato'])
plt.title('Vehicle Maintenance Status by Zone')
plt.ylabel('Vehicles')
plt.xlabel('Zone')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

## 3. Delivery duration

In [ ]:
deliv = deliveries.copy()
deliv['actual_hours'] = (deliv['delivery_completed_at'] - deliv['dispatch_time']).dt.total_seconds() / 3600
merged = deliv.dropna(subset=['actual_hours', 'promised_window_hours'])
merged = merged[merged['actual_hours'] > 0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merged['promised_window_hours'], merged['actual_hours'], alpha=0.4, color='steelblue', s=20)
max_v = max(merged['promised_window_hours'].max(), merged['actual_hours'].max())
ax.plot([0, max_v], [0, max_v], color='red', linestyle='--', label='Promised = Actual')
ax.set_xlabel('Promised Window (hours)')
ax.set_ylabel('Actual Duration (hours)')
ax.set_title('Actual Delivery Duration vs Promised Window')
ax.legend()
plt.tight_layout()
plt.show()

breached = merged[merged['actual_hours'] > merged['promised_window_hours']]
print(f'Window breaches: {len(breached)} of {len(merged)} ({len(breached)/len(merged)*100:.1f}%)')

In [ ]:
proof_outcome = pd.crosstab(deliveries['proof_of_completion_missing'],
                            deliveries['delivery_status'], normalize='index') * 100
proof_outcome.index = ['Proof Present', 'Proof Missing']
print(proof_outcome.round(1))

ax = proof_outcome.plot(kind='bar', figsize=(9, 5), color=['steelblue', '#FCD34D', 'tomato'])
plt.title('Delivery Outcome by Proof of Completion')
plt.ylabel('Percent (%)')
plt.xlabel('')
plt.xticks(rotation=0)
plt.legend(title='Outcome')
plt.tight_layout()
plt.show()

## 4. App events

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
zones_ordered = sorted(app_events['zone_context'].dropna().unique())
data = [app_events[app_events['zone_context'] == z]['api_latency_ms'].dropna() for z in zones_ordered]

ax.boxplot(data, labels=zones_ordered, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_xlabel('Zone')
ax.set_ylabel('API Latency (ms)')
ax.set_title('API Latency by Zone')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(app_events.groupby('zone_context')['api_latency_ms'].median().round(1).sort_values(ascending=False))

In [ ]:
success = app_events.groupby('event_type').agg(
    total=('event_id', 'count'),
    successes=('success_flag', 'sum')
)
success['success_rate_pct'] = (success['successes'] / success['total'] * 100).round(1)
success = success.sort_values('success_rate_pct')
print(success)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(success.index, success['success_rate_pct'], color='steelblue')
ax.axvline(success['success_rate_pct'].mean(), color='red', linestyle='--',
           label=f'Mean: {success["success_rate_pct"].mean():.1f}%')
ax.set_xlabel('Success Rate (%)')
ax.set_title('App Event Success Rate by Event Type')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Cross-dataset

In [ ]:
inc_hub = (incidents
    .merge(deliveries[['delivery_id', 'hub_id']], on='delivery_id', how='left')
    .merge(hubs[['hub_id', 'hub_name']], on='hub_id', how='left'))

inc_pivot = pd.crosstab(inc_hub['hub_name'], inc_hub['incident_type'])
print(inc_pivot)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(inc_pivot, annot=True, fmt='d', cmap='Reds', ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Incident Types by Hub')
plt.tight_layout()
plt.show()

In [ ]:
merged_priority = deliveries.merge(orders[['order_id', 'priority_level']], on='order_id', how='inner')

priority_outcome = pd.crosstab(merged_priority['priority_level'],
                                merged_priority['delivery_status'],
                                normalize='index') * 100
priority_outcome = priority_outcome.reindex(['Low', 'Medium', 'High', 'Critical'])
print(priority_outcome.round(1))

ax = priority_outcome.plot(kind='bar', figsize=(10, 5), color=['steelblue', '#FCD34D', 'tomato'])
plt.title('Delivery Outcome by Order Priority')
plt.ylabel('Percent (%)')
plt.xlabel('Priority Level')
plt.xticks(rotation=0)
plt.legend(title='Outcome')
plt.tight_layout()
plt.show()